# 🎬 ReviewSense: Movie Review Sentiment Classifier
### End-to-End NLP Pipeline: EDA, TF-IDF, Logistic Regression & Naive Bayes
---
**Author:** Senior ML & NLP Engineer  
**Dataset:** IMDb 50,000 Movie Reviews (`dataset/IMDB Dataset.csv`)  
**Domain:** Natural Language Processing (Sentiment Analysis)  
**Objective:** Accurately classify user reviews into **Positive** (Standing Ovation) or **Negative** (Walkout) while handling subtle linguistic edge-cases like negations.

## 1. Environment Setup & Imports
Importing core scientific computing, NLP, visualization, and modeling libraries.

In [1]:
import os
import sys
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Add workspace root to sys.path to import shared preprocessing
sys.path.append(os.path.abspath('..'))
from preprocessing import clean_text

# Plot styling
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='muted')
print('✔ Libraries imported successfully.')

## 2. Dataset Inspection & Exploratory Data Analysis (EDA)
Loading the dataset from `dataset/IMDB Dataset.csv` and analyzing structure, null values, duplicates, and class balance.

In [2]:
dataset_path = '../dataset/IMDB Dataset.csv'
df_raw = pd.read_csv(dataset_path)
print(f'Raw dataset shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
print(f'Missing values:\n{df_raw.isnull().sum()}')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
print(f'\nClass balance:\n{df_raw["sentiment"].value_counts()}')
df_raw.head()

## 3. Data Cleaning & Deduplication
Dropping duplicate reviews and missing records before splitting to prevent data leakage and train-test contamination.

In [3]:
df = df_raw.dropna().drop_duplicates().copy()
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(f'Cleaned dataset shape: {df.shape} (Removed {len(df_raw) - len(df)} duplicates)')
print('Class distribution after deduplication:')
print(df['sentiment'].value_counts(normalize=True) * 100)

## 4. Text Preprocessing & Negation Preservation
Using the single shared `clean_text()` pipeline from `preprocessing.py`.
- Lowercasing
- Stripping HTML markup (`<br />`)
- Expanding negation contractions (`didn't` -> `did not`)
- Filtering stop words while strictly preserving critical negation words (`not`, `no`, `never`, `nothing`).

In [4]:
sample_review = df['review'].iloc[0]
print('--- RAW SAMPLE REVIEW ---')
print(sample_review[:250] + '...')
print('\n--- CLEANED REVIEW ---')
print(clean_text(sample_review)[:250] + '...')

print('\nNormalizing entire corpus...')
t0 = time.time()
df['cleaned_review'] = df['review'].apply(clean_text)
print(f'Corpus cleaned in {time.time() - t0:.2f} seconds.')

## 5. Stratified 80/20 Train-Test Split
Splitting the dataset into 80% training and 20% held-out test set, stratified by sentiment class with `random_state=42`.

In [5]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_review'].tolist(),
    df['label'].tolist(),
    test_size=0.20,
    random_state=42,
    stratify=df['label'].tolist()
)
y_train = np.array(y_train)
y_test = np.array(y_test)
print(f'Training set samples: {len(X_train_raw):,}')
print(f'Testing set samples:  {len(X_test_raw):,}')

## 6. TF-IDF Feature Extraction (Preventing Data Leakage)
Fitting `TfidfVectorizer` strictly on `X_train_raw`, and then transforming `X_test_raw`.

In [6]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    sublinear_tf=True
)
X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)
print(f'Extracted feature vocabulary size: {len(vectorizer.vocabulary_):,}')
print(f'Train matrix shape: {X_train_tfidf.shape}')
print(f'Test matrix shape:  {X_test_tfidf.shape}')

## 7. Model Training & Held-Out Test Set Evaluation
Training and comparing Logistic Regression (`max_iter=1000`) and Multinomial Naive Bayes.

In [7]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'MultinomialNB': MultinomialNB(alpha=1.0)
}

results = {}
for name, clf in models.items():
    t0 = time.time()
    clf.fit(X_train_tfidf, y_train)
    train_time = time.time() - t0
    y_pred = clf.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        'model': clf,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'confusion_matrix': cm,
        'train_time': train_time
    }
    print(f'=== {name} ===')
    print(f'Accuracy:  {acc*100:.2f}%')
    print(f'Precision: {prec*100:.2f}%')
    print(f'Recall:    {rec*100:.2f}%')
    print(f'F1-Score:  {f1*100:.2f}%')
    print(f'Trained in {train_time:.2f}s\n')

## 8. Confusion Matrix Visualization
Plotting confusion matrices for both classifiers using dark theme palette.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(
        res['confusion_matrix'],
        annot=True,
        fmt='d',
        cmap='YlOrBr',
        xticklabels=['Negative', 'Positive'],
        yticklabels=['Negative', 'Positive'],
        ax=ax
    )
    ax.set_title(f"{name} (F1: {res['f1']*100:.2f}%)", fontsize=13, weight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.show()

## 9. Inference on Edge Cases & Negations
Validating model predictions on real-world test inputs, specifically verifying handling of subtle negation.

In [9]:
best_clf = results['LogisticRegression']['model']
test_reviews = [
    'A true cinematic triumph with profound depth and visionary storytelling.',
    'An unmitigated catastrophe. Terrible acting and a painful script.',
    'The movie was not good at all, despite what the trailers claimed.',
    'I really wanted to love this, but I did not enjoy the second half.'
]

print('--- INFERENCE PREVIEW ---')
for r in test_reviews:
    cleaned = clean_text(r)
    vec = vectorizer.transform([cleaned])
    pred = best_clf.predict(vec)[0]
    prob = best_clf.predict_proba(vec)[0]
    verdict = 'POSITIVE' if pred == 1 else 'NEGATIVE'
    conf = prob[pred] * 100
    print(f'Review: "{r}"')
    print(f'Verdict: {verdict} ({conf:.2f}% confidence) | Probabilities: Neg={prob[0]*100:.1f}%, Pos={prob[1]*100:.1f}%\n')